# Explore here

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import classification_report

In [14]:
# Cargar datos
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
data = pd.read_csv(url)
data.head()


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [15]:
data["polarity"].value_counts(normalize=True)

polarity
0    0.655443
1    0.344557
Name: proportion, dtype: float64

In [16]:
# Dejar solo lo importante
data = data.drop("package_name", axis=1)
data.head()

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0


In [17]:
data["review"] = data["review"].str.strip().str.lower()

In [18]:
# Separar texto y resultado
X = data["review"]      # reseñas
y = data["polarity"]    # 0 o 1

In [19]:
# Separar datos para entrenar y probar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y,random_state=42
)

In [20]:
# Convertir palabras en números
vec_model = CountVectorizer(stop_words="english")

X_train_vec = vec_model.fit_transform(X_train)
X_test_vec = vec_model.transform(X_test)

# Analizamos los 3 modelos

In [21]:
model_multi = MultinomialNB()
model_multi.fit(X_train_vec, y_train)

y_pred_multi = model_multi.predict(X_test_vec)

print("MultinomialNB")
print(classification_report(y_test, y_pred_multi))

MultinomialNB
              precision    recall  f1-score   support

           0       0.84      0.96      0.90       117
           1       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179



In [22]:
from sklearn.naive_bayes import GaussianNB

model_gauss = GaussianNB()
model_gauss.fit(X_train_vec.toarray(), y_train)

y_pred_gauss = model_gauss.predict(X_test_vec.toarray())

print("GaussianNB")
print(classification_report(y_test, y_pred_gauss))

GaussianNB
              precision    recall  f1-score   support

           0       0.84      0.89      0.86       117
           1       0.76      0.68      0.72        62

    accuracy                           0.82       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.81      0.82      0.81       179



In [23]:
from sklearn.naive_bayes import BernoulliNB

model_bern = BernoulliNB()
model_bern.fit(X_train_vec, y_train)

y_pred_bern = model_bern.predict(X_test_vec)

print("BernoulliNB")
print(classification_report(y_test, y_pred_bern))

BernoulliNB
              precision    recall  f1-score   support

           0       0.76      0.97      0.85       117
           1       0.87      0.44      0.58        62

    accuracy                           0.78       179
   macro avg       0.82      0.70      0.72       179
weighted avg       0.80      0.78      0.76       179



MultinomialNB es el más adecuado porque trabaja con frecuencia de palabras.

# Optimización de Modelo

In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_vec, y_train)

y_pred_rf = rf_model.predict(X_test_vec)

y_proba = model_multi.predict_proba(X_test_vec)[:, 1]

# ROC AUC
roc_auc = roc_auc_score(y_test, y_proba)

print("Random Forest")
print("ROC AUC:", roc_auc)
print(classification_report(y_test, y_pred_rf))

Random Forest
ROC AUC: 0.8986765922249793
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       117
           1       0.79      0.66      0.72        62

    accuracy                           0.82       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.82      0.82      0.82       179



In [28]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42
)

random_search.fit(X_train_vec, y_train)

best_rf = random_search.best_estimator_

In [31]:
y_pred_rf = best_rf.predict(X_test_vec)

# ROC AUC
roc_auc = roc_auc_score(y_test, y_proba)

y_proba_rf = best_rf.predict_proba(X_test_vec)[:, 1]

roc_auc_rf = roc_auc_score(y_test, y_proba_rf)

print("Random Forest Optimizado")
print("ROC AUC:", roc_auc_rf)
print(classification_report(y_test, y_pred_rf))

Random Forest Optimizado
ROC AUC: 0.8919216983733113
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       117
           1       0.79      0.66      0.72        62

    accuracy                           0.82       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.82      0.82      0.82       179



In [33]:
# Ajustamos threshold
y_pred_custom = (y_proba > 0.4).astype(int)

print("Random Forest con threshold 0.4")
print(classification_report(y_test, y_pred_custom))

Random Forest con threshold 0.4
              precision    recall  f1-score   support

           0       0.89      0.83      0.86       117
           1       0.71      0.81      0.76        62

    accuracy                           0.82       179
   macro avg       0.80      0.82      0.81       179
weighted avg       0.83      0.82      0.82       179



# Conclusión:

Tras el preprocesamiento del texto y su transformación a variables numéricas mediante CountVectorizer, se entrenaron distintas variantes de Naive Bayes (Gaussian, Multinomial y Bernoulli), siendo Multinomial Naive Bayes el modelo que presentó el mejor desempeño.
Por otro lado, el modelo Random Forest, incluso tras su optimización mediante búsqueda de hiperparámetros, no logró superar el rendimiento de Naive Bayes en métricas clave como accuracy y F1 score. Sin embargo, presentó un buen valor de ROC AUC, indicando una adecuada capacidad de discriminación entre clases.
En conclusión, Multinomial Naive Bayes se posiciona como la mejor alternativa para este problema, destacando por su simplicidad, rapidez y alto rendimiento en tareas de procesamiento de lenguaje natural.